# ScienceQA — Ensemble Inference (multi-adapter + TTA)

Loads multiple LoRA adapters trained by `overnight_train.py`, scores each on the test
set with choice-permutation TTA, **averages the per-letter logits across all
adapters and all permutations**, and writes a submission.

This is the morning-after notebook. Set `ADAPTER_DIRS` below to the Drive paths
listed in `OVERNIGHT_SUMMARY.json`.


In [1]:
%pip install -q transformers==4.57.6 peft==0.18.1 accelerate==1.0.1 pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 143.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# 1. Define paths
# UPDATE THIS to match where the 'images' folder is in your Google Drive
drive_images_path = '/content/drive/MyDrive/images/test'

local_data_dir = '/content/data'
local_images_path = '/content/data/images/test'

# Ensure the local data directory exists (where the CSVs should be)
os.makedirs(local_data_dir, exist_ok=True)

# 2. Copy images to local Colab storage for faster training I/O
if not os.path.exists(local_images_path):
    print(f"Copying images from {drive_images_path} to {local_images_path}...")
    print("This might take a few minutes depending on the size, but drastically speeds up training.")
    !cp -r "{drive_images_path}" "{local_data_dir}/"
    print("Copy complete! ✅")
else:
    print("Images already exist locally! ✅")


Copying images from /content/drive/MyDrive/images/test to /content/data/images/test...
This might take a few minutes depending on the size, but drastically speeds up training.
Copy complete! ✅


In [4]:
import os, json, gc, random, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ════════════ EDIT ME ════════════════════════════════════════════════════
DATA_DIR  = Path("/content/data")
OCR_JSON  = Path("/content/drive/MyDrive/scienceqa_run_final_overnight/ocr_text.json")
OUT_CSV   = Path("/content/drive/MyDrive/scienceqa_run_final_overnight/submission_ocr_v4.csv")

ADAPTERS = [
    # (Path("/content/drive/MyDrive/scienceqa_run_final_overnight/seed99_res512_OCR_8epp"),   512, True,  1.0),  # Run E
    (Path("/content/drive/MyDrive/scienceqa_run_final_overnight/seed11_res512_OCR_8ep"),   512, True,  1.0),  # Run F

]
TTA_PERMUTATIONS  = 8           # match original — DO NOT bump
EVAL_BATCH        = 8           # match original — DO NOT bump (or use 16 max)
OCR_MAX_CHARS     = 500
MAX_CONTEXT_CHARS = 1200
MODEL_ID          = "HuggingFaceTB/SmolVLM-500M-Instruct"
# ═════════════════════════════════════════════════════════════════════════

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError: pass

# Sanity
for a in ADAPTERS:
    assert a[0].exists(), f"missing: {a[0]}"
assert OCR_JSON.exists(), f"OCR JSON missing: {OCR_JSON}"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"adapters: {len(ADAPTERS)}  TTA={TTA_PERMUTATIONS}  device={device}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
adapters: 1  TTA=8  device=cuda


In [5]:
# ── Load test + OCR ──────────────────────────────────────────────────────
test_df = pd.read_csv(DATA_DIR / "test.csv")
test_df["choices"] = test_df["choices"].apply(json.loads)
OCR_TEXT = json.loads(OCR_JSON.read_text())
n_ocr = sum(1 for v in OCR_TEXT.values() if v.strip())
print(f"test rows: {len(test_df)}  ocr cache: {n_ocr}/{len(OCR_TEXT)} have text")

CHOICE_LETTERS = "ABCDEFGHIJ"

def _truncate(s, n):
    s = str(s).strip()
    return s if len(s) <= n else s[:n-1] + "…"

def build_user_text(row, choices, *, use_ocr):
    """Must mirror EXACTLY the training-time prompt format for each adapter."""
    parts = []
    lec = row.get("lecture")
    if isinstance(lec, str) and lec.strip(): parts.append(_truncate(lec, MAX_CONTEXT_CHARS))
    hint = row.get("hint")
    if isinstance(hint, str) and hint.strip(): parts.append(_truncate(hint, MAX_CONTEXT_CHARS//2))
    if use_ocr:
        ocr = (OCR_TEXT.get(row["id"], "") or "").strip()
        if ocr:
            parts.append(f"Text extracted from image (via OCR): {ocr[:OCR_MAX_CHARS]}")
    ctx = "\n".join(parts)
    cl  = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    text = ""
    if ctx: text += f"Context:\n{ctx}\n\n"
    text += f"Question: {row['question']}\n\nChoices:\n{cl}\n\nAnswer with a single letter only."
    return text

def build_messages(row, choices, *, use_ocr):
    return [{"role":"user","content":[
        {"type":"image"},
        {"type":"text","text":build_user_text(row, choices, use_ocr=use_ocr)},
    ]}]

def load_image(rel, img_size):
    im = Image.open(DATA_DIR / rel).convert("RGB")
    w, h = im.size; s = img_size / max(w, h)
    if s < 1.0: im = im.resize((max(1,int(w*s)), max(1,int(h*s))), Image.BICUBIC)
    return im

test rows: 1008  ocr cache: 2912/5165 have text


In [6]:
# Pre-build row dicts
test_rows = [{
    "id": r["id"], "image_path": r["image_path"], "question": r["question"],
    "choices": r["choices"], "lecture": r.get("lecture"), "hint": r.get("hint"),
} for _, r in test_df.iterrows()]


# ── Load base model ONCE (saves a few minutes per adapter) ───────────────
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel

print("loading base model + processor (once)...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True,
)
base.eval(); base.config.use_cache = True

def letter_id(L):
    ids = processor.tokenizer(L, add_special_tokens=False).input_ids
    return ids[-1] if len(ids) >= 1 else processor.tokenizer(" "+L, add_special_tokens=False).input_ids[-1]
LIDS = {L: letter_id(L) for L in CHOICE_LETTERS}

loading base model + processor (once)...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

In [7]:
# ── Score one adapter end-to-end ─────────────────────────────────────────
def score_with_adapter(adapter_dir, img_size, use_ocr, tta_perms):
    print(f"\n=== {adapter_dir.name}  img={img_size}  ocr={use_ocr}  TTA={tta_perms} ===")
    t0 = time.time()
    # Load adapter onto base
    model = PeftModel.from_pretrained(base, str(adapter_dir))
    model.eval(); model.config.use_cache = True

    images = [load_image(r["image_path"], img_size) for r in test_rows]
    summed = [np.zeros(len(r["choices"]), dtype=np.float64) for r in test_rows]
    rng = np.random.RandomState(0)

    @torch.inference_mode()
    def score_pass(perms):
        for start in range(0, len(test_rows), EVAL_BATCH):
            sl = slice(start, min(start+EVAL_BATCH, len(test_rows)))
            rows_b = test_rows[sl]; imgs_b = images[sl]; perms_b = perms[sl]
            shuffled = [[r["choices"][i] for i in p] for r, p in zip(rows_b, perms_b)]
            prompts = [processor.apply_chat_template(
                build_messages(r, ch, use_ocr=use_ocr), add_generation_prompt=True
            ) for r, ch in zip(rows_b, shuffled)]
            inp = processor(text=prompts, images=[[im] for im in imgs_b],
                            return_tensors="pt", padding=True)
            inp = {k:(v.to(model.device) if torch.is_tensor(v) else v) for k,v in inp.items()}
            out = model(**inp); last = out.logits[:, -1, :]
            for i, p in enumerate(perms_b):
                ids = [LIDS[CHOICE_LETTERS[k]] for k in range(len(p))]
                row_logits = last[i, ids].float().cpu().numpy()
                gi = start + i
                for j, orig_idx in enumerate(p):
                    summed[gi][orig_idx] += float(row_logits[j])

    for k in range(tta_perms):
        perms = [(np.arange(len(r["choices"])) if k == 0 else rng.permutation(len(r["choices"])))
                 for r in test_rows]
        score_pass(perms)
        print(f"  pass {k+1}/{tta_perms}  elapsed={(time.time()-t0)/60:.1f}m")

    # average over passes
    out = [s / tta_perms for s in summed]
    # Unload adapter to free memory and prep for next
    model = model.unload() if hasattr(model, "unload") else model
    del model; gc.collect(); torch.cuda.empty_cache()
    return out


## Score each adapter, then average across adapters

In [8]:
# ── Score all adapters, weighted ensemble ────────────────────────────────
final = [None] * len(test_rows)
total_w = sum(a[3] for a in ADAPTERS)
for adapter_dir, img_size, use_ocr, w in ADAPTERS:
    logits_list = score_with_adapter(adapter_dir, img_size, use_ocr, TTA_PERMUTATIONS)
    for i, lg in enumerate(logits_list):
        if final[i] is None: final[i] = np.zeros_like(lg)
        final[i] += (w / total_w) * lg

preds = [int(np.argmax(s)) for s in final]
print(f"\npreds done: {len(preds)}")



=== seed11_res512_OCR_8ep  img=512  ocr=True  TTA=8 ===
  pass 1/8  elapsed=11.6m
  pass 2/8  elapsed=23.1m
  pass 3/8  elapsed=34.7m
  pass 4/8  elapsed=46.2m
  pass 5/8  elapsed=57.7m
  pass 6/8  elapsed=69.3m
  pass 7/8  elapsed=80.8m
  pass 8/8  elapsed=92.3m

preds done: 1008


In [9]:
sub = pd.DataFrame({"id":[r["id"] for r in test_rows], "answer": preds})
assert list(sub.columns) == ["id","answer"]
assert len(sub) == len(test_df)
assert set(sub["id"]) == set(test_df["id"])
assert sub["answer"].dtype.kind in ("i","u")
for r, p in zip(test_rows, preds):
    assert 0 <= p < len(r["choices"])
sub.to_csv(OUT_CSV, index=False)
print(f"\n✅ wrote {OUT_CSV}  ({len(sub)} rows)")
print("\nAnswer distribution:")
print(sub["answer"].value_counts().sort_index())



✅ wrote /content/drive/MyDrive/scienceqa_run_final_overnight/submission_ocr_v4.csv  (1008 rows)

Answer distribution:
answer
0    366
1    350
2    214
3     70
4      8
Name: count, dtype: int64


---
**Done.** Submit `submission.csv`.

Sanity checks:
- If predictions are nearly all `0`, the chat template / letter token IDs are off.
- If accuracy on train sample looks reasonable but submission scores low on Kaggle, you may have a path mismatch (check `image_path` resolves correctly).


In [ ]:
# ── 10) Disconnect runtime ───────────────────────────────────────────────
import time
print("Sleeping 60s to flush Drive…")

time.sleep(60)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print("runtime.unassign failed:", e); os._exit(0)


Sleeping 60s to flush Drive…
